# 第七轮结果经济解释
只读取最终 Qlib Recorder300530137db54b6c807c39ebb9e8c331 保存结果；没有新的模型拟合、策略路径或阈值搜索。

In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
ROOT=Path('/Users/rziiiii/dev/workspace-worktrees/constrained-net-excess-20261003')
CARD=ROOT/'research/constrained_net_excess_20261003'
summary=json.loads((CARD/'summary.json').read_text())
assert summary['recorder_id']=='300530137db54b6c807c39ebb9e8c331'
qlib.init(provider_uri=str(ROOT/summary['provider_uri']),region='us',kernels=1)
names=['comparison','annual','gates','seed_metrics','training_diagnostics','ensemble_audits','controls','attribution','window_metrics','portfolio_replays','bootstrap']
with R.uri_context(summary['recorder_uri']):
    recorder=R.get_recorder(experiment_name=summary['experiment'],recorder_id=summary['recorder_id'])
    assert recorder.client.get_run(recorder.id).info.status=='FINISHED'
    data={name:recorder.load_object(f'analysis/{name}.pkl') for name in names}
    artifact_hashes={name:hashlib.sha256((Path(recorder.get_local_dir())/f'artifacts/analysis/{name}.pkl').read_bytes()).hexdigest() for name in names}
def records(frame):
    return json.loads(frame.to_json(orient='records',date_format='iso',double_precision=15))
comparison=data['comparison']; annual=data['annual']; paths=data['portfolio_replays']; training=pd.DataFrame(data['training_diagnostics']); ensemble=pd.DataFrame(data['ensemble_audits'])
metrics=['variant','segment','cost_bps','sessions','return_pct','benchmark_return_pct','relative_wealth','annualized_relative_return_pct','max_drawdown_pct','benchmark_max_drawdown_pct','cvar5_pct','benchmark_cvar5_pct','worst_day_pct','benchmark_worst_day_pct','worst_5d_pct','benchmark_worst_5d_pct','turnover','trade_days','sum_session_fee_rates']
exposure=[]
for (variant,cost),part in paths.loc[paths.segment.eq('development') & paths.seed.eq('ensemble')].groupby(['variant','cost_bps']):
    exposure.append({'variant':variant,'cost_bps':cost,'avg_soxl':part.soxl_weight.mean(),'avg_soxx':part.soxx_weight.mean(),'avg_cash':part.cash_weight.mean(),'turnover_per_year':part.turnover.sum()*252/len(part),'all_trades':int(part.execute_open_trade.sum()),'anchor_sessions':int(part.ensemble_anchor_selected.sum()),'first_trade':part.trade_date.min().isoformat(),'last_exit':part.exit_date.max().isoformat()})
annual_dev=annual[annual.segment.eq('development')]
annual_details=annual_dev[['variant','cost_bps','year','relative_return_pct','log_excess_sum','return_pct','benchmark_return_pct','max_drawdown_pct','benchmark_max_drawdown_pct','cvar5_pct','benchmark_cvar5_pct']]
training_dev=training[training.year.le(2022)]
train_summary=[]
for variant,part in training_dev.groupby('variant'):
    train_summary.append({'variant':variant,'seed_fits':len(part),'null_selected':int(part.null_selected.sum()),'failure_count':int(part.failure.notna().sum()),'training_feasibility_all':bool(part.training_feasibility_passed.all()),'F_min':part.centered_worst_utility.min(),'F_max':part.centered_worst_utility.max(),'legacy_J_negative':int(part.legacy_J.lt(0).sum()),'selected_epoch_min':min([float(v) for v in part.selected_epoch if v!='anchor'],default=None),'selected_epoch_max':max([float(v) for v in part.selected_epoch if v!='anchor'],default=None)})
ensemble_cols=['variant','year','failure','ensemble_F','ensemble_feasible','ensemble_rejected_for_training_utility_or_risk','anchor_selected','F','legacy_J','training_feasibility_passed']
window_cols=['variant','cost_bps','start_position','sessions','start_signal','last_exit','return_pct','benchmark_return_pct','relative_return_pct','max_drawdown_pct','benchmark_max_drawdown_pct','turnover','trade_days']
window_dates=[]
for (variant,cost),part in paths.loc[paths.segment.eq('reused_diagnostic') & paths.seed.eq('ensemble') & paths.trade_date.ge('2026-08-03')].groupby(['variant','cost_bps']):
    row=part.iloc[0]
    window_dates.append({'variant':variant,'cost_bps':cost,'first_signal':row.signal_date.isoformat(),'first_trade':row.trade_date.isoformat(),'last_exit':part.exit_date.max().isoformat(),'sessions':len(part),'first_carried_holdings':[row.pretrade_soxl_weight,row.pretrade_soxx_weight,row.pretrade_cash_weight],'first_target':[row.soxl_weight,row.soxx_weight,row.cash_weight]})
report={'source':{'recorder_id':summary['recorder_id'],'plan_sha256':summary['plan_sha256'],'recorder_status':'FINISHED','saved_artifact_sha256':artifact_hashes},'comparison':records(comparison[metrics]),'annual_development':records(annual_details),'gates':records(data['gates']),'attribution_full':records(data['attribution'][data['attribution'].signal_year.eq('all')]),'seed_metrics':records(data['seed_metrics'][['variant','seed','cost_bps','relative_wealth','max_drawdown_pct','cvar5_pct']]),'exposures':exposure,'training_summary':train_summary,'training_development':records(training_dev),'ensemble_audits':records(ensemble[ensemble_cols]),'window_metrics':records(data['window_metrics'][window_cols]),'window_dates':window_dates,'bootstrap':records(data['bootstrap'])}
Path('/tmp/constrained_net_excess_economic_review.json').write_text(json.dumps(report,indent=2,allow_nan=False,default=str))
for name in ['comparison','annual_development','attribution_full','exposures','training_summary','ensemble_audits','window_metrics','gates','seed_metrics']:
    print(name);print(pd.DataFrame(report[name]).to_string(index=False))


[15104:MainThread](2026-10-03 02:53:11,837) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[15104:MainThread](2026-10-03 02:53:11,838) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[15104:MainThread](2026-10-03 02:53:11,838) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-worktrees/constrained-net-excess-20261003/data/qlib/published_futu_affine_20260925_v1')}


comparison
                        variant           segment  cost_bps  sessions  return_pct  benchmark_return_pct  relative_wealth  annualized_relative_return_pct  max_drawdown_pct  benchmark_max_drawdown_pct  cvar5_pct  benchmark_cvar5_pct  worst_day_pct  benchmark_worst_day_pct  worst_5d_pct  benchmark_worst_5d_pct  turnover  trade_days  sum_session_fee_rates
                       constant       development      10.0      1762  325.747413            325.747413         1.000000                   -6.810000e-13        -47.373291                  -47.373291  -4.416437            -4.416437      -9.324950                -9.324950    -17.254827              -17.254827  0.999001           1               0.000999
                       constant       development      25.0      1762  325.110384            325.110384         1.000000                    0.000000e+00        -47.373291                  -47.373291  -4.418079            -4.418079      -9.324950                -9.324950    -17.254

# 第七轮：训练约束已对齐，稳定净超额仍未成立

证据：最终且 FINISHED 的 Qlib Recorder `300530137db54b6c807c39ebb9e8c331`；登记计划 `da17267d7d21ccb2589bc799efe33ed9149d8b4c7c0b5371aadd4b429227901d`。本说明只读取保存的 analysis artifacts，没有新拟合、阈值搜索、其他策略路径或修改模型选择。可执行复核本 `/tmp/constrained_net_excess_economic_review.ipynb`；原始指标和 artifact SHA 在同名 `.json`。

## 已修复的具体问题

两个带信息输入的组，在开发期全部42个种子拟合和14个三种子集合中，都找到了正 F、同时满足两档成本下 MaxDD/CVaR 四项训练约束的状态；没有条件模型退回空模型，没有数值失败。常量组21/21种子及7/7集合选择精确 SOXX，属于合法的“未找到改进”，不能算超额或故障。

price 的12/21种子、price_vol 的8/21种子保留模型仍有负的旧季度 RMS 效用 J。它们具有正的费用后平均净超额和训练尾部可行性，说明旧 J 的额外否决确实会排除其中一部分。此次同时改了目标与约束，不能把全部改进单独归因于去掉 RMS 项。

因此原来“沿现有训练轨迹找不到合格条件模型”的障碍在本轮已解决。全局最优、有限步优化收敛，以及未来收益/风险仍未得到证明。

## 开发期：2016–2022 信号，最后于2023-01-04开盘结算

相对終值表示策略终值除以同成本 SOXX 终值；不是收益率相减。每个组均有1,762个交易区间。

|组|10bp相对终值|25bp相对终值|正超额年份10/25bp|MaxDD 10/25bp|CVaR5 10/25bp|
|---|---:|---:|---:|---:|---:|
|constant|1.00000|1.00000|0/7、0/7|−47.373%、−47.373%|−4.416%、−4.418%|
|price|1.09577|1.01110|2/7、2/7|−45.419%、−46.102%|−4.335%、−4.340%|
|price_vol|1.12713|1.03285|4/7、2/7|−45.837%、−46.528%|−4.396%、−4.402%|

三个组均未取得历史资格。price_vol 是原规则选出的诊断对象，没有修改选择。

尾部改善在连续开发路径上成立：两个条件组的整体 MaxDD/CVaR 都通过原 SOXX 门。price_vol 的风险空间很窄，25bp CVaR 只比 SOXX 改善约0.0162个百分点，MaxDD改善约0.8455个百分点。该门没有要求所有风险统计都优于 SOXX：price_vol 的最差单日−10.693%仍差于−9.325%，最差五日−20.208%也差于−17.255%；各单年仍可能有更差尾部。

price_vol 在10bp下四个正年份为2016、2018、2019、2022；25bp下只剩2016和2022。25bp逐年相对收益为：+6.585%、−1.015%、−0.435%、−0.067%、−1.169%、−3.618%、+3.293%。2017–2021累计净对数超额为−0.06384；2016与2022合计+0.09617。去掉2016后，剩余年份净对数超额变为−0.03145，故高成本的删一年稳健性失败。两种固定区块长度的家族调整 bootstrap 下界也均不大于零，price_vol最差年化下界−3.381%。

## 费用、持仓、种子与 alpha

price/price_vol平均目标分别为 SOXL12.69%/13.57%、SOXX61.62%/60.23%、现金25.68%/26.21%；不是固定现金规则，而是保存策略路径的均值。实际 β 分别约0.9886和1.0041，较早期通过显著提高市场暴露取得大回报的情况已有改善。

但两组仍在1,762/1,762天交易，年化总成交额分别约7.81和8.47倍当期净值。把成本从10提高到25bp，price 的累计净对数超额减少87.9%，price_vol减少73.0%。price_vol相对年化收益从1.726%降至0.463%；price从1.317%降至0.158%。这是薄弱且费用敏感的优势，不是“几乎没有成本”。

HAC20回归中，price_vol年化算术 alpha 点估计10bp为1.720%，95%区间[−1.223%,4.663%]；25bp为0.471%，区间[−2.462%,3.405%]。price相应为1.559%和0.409%，区间也都跨零。β接近1有助于排除简单高β叙事，但这些区间没有建立稳定信息优势，且不包含模型筛选不确定性。

price_vol三种子25bp相对终值分别为1.06265、1.00702、1.01235，三个都正，但幅度分散；seed71的开发路径25bp MaxDD−47.779%、CVaR−4.487%均差于SOXX。原种子门只要求共同的至少两个种子在两成本下有正超额，并不证明每个种子的样本外风险都受控。price只有17、41两个种子两成本皆正，71在25bp相对终值0.97683。无证据表明结果由条件组空模型切换支撑；所有条件开发折都保留非空模型。

## 已查看的2023以后诊断与用户8月3日窗口

2023以后完整诊断也没有支持稳定性：price_vol年更模式相对终值10/25bp为0.89410/0.84123；冻结2022年底参数模式为0.86556/0.80473。年更模式最大回撤−45.142%/−45.742%，均差于同窗SOXX−42.789%，虽然CVaR较好。该历史已经查看过，不能充当新的独立样本。

用户窗口采用实际交易日，从2026-08-03到09-25开盘，共38个区间，首信号2026-07-31。以下只比较一致终点；不能与此前截至09-30收盘的结果混比。持续持仓来自研究模拟的2023年初起点，并非用户真实账户。

|模式及起点|10bp策略 / SOXX收益|25bp策略 / SOXX收益|10bp最大回撤策略 / SOXX|
|---|---:|---:|---:|
|年更、持续持仓|13.297% / 15.843%|13.035% / 15.843%|−9.882% / −10.984%|
|年更、8/3现金起步|13.210% / 15.727%|12.836% / 15.554%|−9.882% / −10.984%|
|冻结、持续持仓|14.568% / 15.843%|14.260% / 15.843%|−8.856% / −10.984%|
|冻结、8/3现金起步|14.296% / 15.727%|13.896% / 15.554%|−8.856% / −10.984%|

四个定义下均未跑赢对应 SOXX，回撤有所减轻；38天仍全部交易。

## 一个下一步诊断

建议只对已保存的同一目标/持仓路径，按既定年度拆出“持仓产生的费用前相对收益”和“已发生的换手费用拖累”，同时列训练与相邻开发年，不拟合或重放另一个策略。用精确恒等式：

`log(net_multiplier) = log(gross_multiplier_on_saved_targets) + log(1 - saved_fee_rate)`。

基准也按同样方式拆分；逐日、逐年及全路径必须精确加回当前净超额。这是解释现有路径，不是模拟免费交易、更改目标、搜索交易阈值或证明少交易一定更好。它能检验2017–2021的负超额究竟主要来自持仓时机本身，还是很小的持仓优势被密集调仓消耗；只有前者/后者得到实际数字支持后，才登记单一机制的新假设。当前证据不能把不稳定全归咎于信息无效，也不能归咎于未找到可行解。

原资格决定不变：继续研究，无模型上线替换，无稳定超额或新前向资格声明。
